# 04. Alert scheduling, hindsight oracles and the economic value of alerts

Here I locate where the warning system loses opportunities along the chain gate → score → alert scheduler → trading action, without adding new predictive features.

- I use the one-minute Binance spot BTCUSDT cache of notebook 01 and the price-only crash catalog of notebook 02. I fit the model on 2021, calibrate thresholds on 2022 without event labels and evaluate on 2023 to April 2026.
- One V+F score (volatility and taker-sell ratio, no low-volatility gate) drives four schedulers and a no-alert policy at annual budgets of 60, 120, 240 and 480 alerts (primary 240). An exact integer program gives a hindsight oracle ceiling, and I measure economic value with a stylized hedge after each alert.
- This analysis is retrospective and exploratory; it is not a preregistration.
- I write the results to `systems_engineering_scheduler_oracle_value_v1/`. Some cells of this saved copy have no output; their tables are in that folder.


## Study specification

- Price-only crash catalog: a 4% breach from the trailing 240-minute peak close, 5-minute confirmation, 24-hour refractory period (notebook 02).
- Valid detection: one alert per event in [30, 240] minutes before the first 4% breach. Primary metric: alerts before the first 1% drop; secondary: any valid alert before the breach. The 1% criterion uses future data only to *score*, never to trigger.
- Baseline score: I train `V+F` (volatility and taker-sell ratio) on 2021, without the low-volatility gate, and calibrate score and alert thresholds on 2022 without event labels.
- Policies: calendar-month quota, 30-day rolling quota, within-month pacing, token bucket (refill B per 365 days, burst capacity b) and no alert.
- Budgets: B = 60, 120, 240 and 480 alerts per full year are experimental levels, not estimated optimal budgets. A hard calendar-year cap of B applies to every policy; 2026 is an incomplete year.
- Two arms: (A) I hold the calendar-calibrated threshold constant to isolate scheduler mechanics; (B) I calibrate each scheduler separately to the same 2022 annual alert target.
- Oracle: an exact integer program for hindsight event capture among candidate threshold crossings, with one alert per event and the annual cap of B, plus a version with the calendar-month cap. The oracle is a clairvoyant, false-alarm-free ceiling, not an implementable scheduler and not pure scheduler headroom. Event windows lie at least one cooldown apart.
- Economic action: on each alert I halve a 100%-long BTC exposure for 240 minutes from the next executable minute close, charge entry and exit fees, and compare the result with buy-and-hold and an exposure-matched constant portfolio. This is a stylized simulation, not trading profit and loss.
- Main question: is there an economically or statistically credible pre-onset improvement at a similar realized alert burden?

Synthetic zero-volume gap fills are never executable bars, the scores are causal, and the cooldown carries across calendar boundaries. I pair events and keep the oracle and the real policies separate.

In [2]:
# Run once in Google Colab. Existing Parquet cache is re-used; no Binance redownload/FFT.
try:
    import pyarrow  # noqa
except ImportError:
    %pip -q install pyarrow

from pathlib import Path
from collections import deque, defaultdict
import hashlib, json, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.optimize import milp, Bounds, LinearConstraint, minimize
from scipy.stats import binomtest, spearmanr
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab: update BASE for local data before running production cells')

BASE = Path('/content/drive/MyDrive/btc_lead_cost_results_77months_spot')
# --- SPOT guard: this notebook may only read the spot cache built and fingerprint-verified by notebook 1 ---
import json as _json
_src_file = BASE/'cache'/'DATA_SOURCE.json'
if not _src_file.is_file() or _json.loads(_src_file.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Not a spot cache: {_src_file} is missing or venue!=spot. Run notebook 1 (1_..._SPOT) to the end first.')
print('SPOT cache verified:', _json.loads(_src_file.read_text())['market'])

CACHE = BASE/'cache'
EVENT_CSV = BASE/'event_definition_eligibility_audit'/'price_only_event_catalog.csv'
OUT = BASE/'systems_engineering_scheduler_oracle_value_v1'
OUT.mkdir(parents=True, exist_ok=True)
for fp in [CACHE/'BTCUSDT_1m_regularized.parquet', CACHE/'btc_signals_1m.parquet', EVENT_CSV]:
    if not fp.exists(): raise FileNotFoundError(fp)

# I write all specifications to disk before scoring the evaluation split.
CONFIG = dict(version='v1', primary_budget=240, budgets=[60,120,240,480],
    model='V+F', score_features=['z_vol','tsr'], model_train_year=2021,
    threshold_calibration_year=2022, eval_start='2023-01-01', eval_end_exclusive='2026-05-01',
    event_catalog=str(EVENT_CSV), lead_min=30, lead_max=240,
    cooldown_min=240, preonset_metric='before first 1pct drawdown',
    event_def='4pct 240min peak close / 5min sustain / 24h refractory',
    primary_scheduler='calendar', primary_value_fee_bps=10,
    bucket_burst_rule='ceil(B/48), minimum 2',
    research_status='retrospective exploratory — evaluation periods repeatedly inspected')
with open(OUT/'study_specification.json','w',encoding='utf-8') as f:
    json.dump(CONFIG,f,indent=2,ensure_ascii=False)
sha = hashlib.sha256((OUT/'study_specification.json').read_bytes()).hexdigest()
(OUT/'study_specification.sha256').write_text(sha+'  study_specification.json\n')
print('Study-spec SHA256:',sha)
print('This records the specification at run time.')

# Optional capacity grounding for human-review warning systems; these are scenarios,
# not estimates of actual analyst workflows, and not appropriate for automated hedges.
capacity=[]
for hrs_week in [0.5,1.0,2.0]:
    for mins_review in [5,10,15,30]:
        Bmax=math.floor(52*hrs_week*60/mins_review)
        capacity.append(dict(analyst_hours_allocated_per_week=hrs_week,
            minutes_per_alert_review=mins_review,staff_count=1,
            annual_review_capacity_alerts=Bmax))
cap_scenarios=pd.DataFrame(capacity)
cap_scenarios.to_csv(OUT/'ILLUSTRATIVE_review_capacity_budgets.csv',index=False)
print('Illustrative staffing-to-budget conversion (not actual BTC desk costs):')
display(cap_scenarios)

Mounted at /content/drive
SPOT cache verified: Binance spot BTCUSDT 1m klines
Study-spec SHA256: f5d82820355fe19988c81b2c5f3b69f454986e694031497f315daef61763cc10
This records the specification at run time.
Illustrative staffing-to-budget conversion (not actual BTC desk costs):


,analyst_hours_allocated_per_week,minutes_per_alert_review,staff_count,annual_review_capacity_alerts
0,0.5,5,1,312
1,0.5,10,1,156
2,0.5,15,1,104
3,0.5,30,1,52
4,1.0,5,1,624
5,1.0,10,1,312
6,1.0,15,1,208
7,1.0,30,1,104
8,2.0,5,1,1248
9,2.0,10,1,624


## 1. Event catalog and causal baseline score

In [3]:
btc = pd.read_parquet(CACHE/'BTCUSDT_1m_regularized.parquet', columns=['close','volume']).sort_index()
sig = pd.read_parquet(CACHE/'btc_signals_1m.parquet', columns=['z_vol','tsr','z_hs','price_lowvol']).sort_index()
for d in (btc,sig):
    if not isinstance(d.index,pd.DatetimeIndex) or d.index.tz is None:
        raise ValueError('UTC timezone-aware minute index required')
    d.index=d.index.tz_convert('UTC')
if not btc.index.equals(sig.index): raise ValueError('Cache date indices differ')
if btc.index.has_duplicates or not btc.index.is_monotonic_increasing: raise ValueError('Bad index')
if not np.all(np.diff(btc.index.asi8)==60*10**9):
    raise ValueError('Expected regularized 1-minute bars')

catalog=pd.read_csv(EVENT_CSV)
for col in ['peak_utc','onset_1pct_utc','breach_utc','confirmed_utc']:
    catalog[col]=pd.to_datetime(catalog[col],utc=True)
catalog=catalog.sort_values('breach_utc').reset_index(drop=True)
assert catalog.event_id.is_unique and catalog.breach_utc.is_unique
assert (catalog.confirmed_utc>=catalog.breach_utc).all()

NS=60*10**9; LEAD_MIN=30; LEAD_MAX=240; COOLDOWN_MIN=240
START_TRAIN=pd.Timestamp('2021-01-01',tz='UTC')
START_CAL=pd.Timestamp('2022-01-01',tz='UTC')
START_EVAL=pd.Timestamp('2023-01-01',tz='UTC')
END_EVAL=min(pd.Timestamp('2026-05-01',tz='UTC'),btc.index[-1]+pd.Timedelta(minutes=1))
T=btc.index.asi8
close=btc.close.to_numpy(dtype=np.float64)
volume=btc.volume.to_numpy(dtype=np.float64)
V=sig.z_vol.to_numpy(dtype=np.float64)
F=sig.tsr.to_numpy(dtype=np.float64)
H=sig.z_hs.to_numpy(dtype=np.float64)
valid=np.isfinite(close)&(close>0)&np.isfinite(volume)&(volume>0)&np.isfinite(V)&np.isfinite(F)&np.isfinite(H)&(F>=0)&(F<=1)

# For every t, positive iff an independent breach is in [t+30m,t+240m].
BREACH=catalog.breach_utc.astype('int64').to_numpy()
next_event=np.searchsorted(BREACH,T+LEAD_MIN*NS,'left')
label=np.zeros(len(T),bool);has_next=next_event<len(BREACH)
label[has_next]=(BREACH[next_event[has_next]]<=T[has_next]+LEAD_MAX*NS)
train=valid&(T>=START_TRAIN.value)&(T<(START_CAL-pd.Timedelta(minutes=LEAD_MAX+5)).value)
cal=valid&(T>=START_CAL.value)&(T<START_EVAL.value)
ev=valid&(T>=START_EVAL.value)&(T<END_EVAL.value)
assert train.any() and cal.any() and ev.any()
assert 0<label[train].sum()<train.sum()

# Standardize using training distribution only. The same V+F score drives all schedulers.
clf=make_pipeline(StandardScaler(),LogisticRegression(C=0.2,class_weight='balanced',
                max_iter=400,solver='lbfgs',random_state=17))
X=np.column_stack([V,F])
clf.fit(X[train],label[train].astype('int8'))
T_CAL=T[cal]; SC_CAL=clf.predict_proba(X[cal])[:,1]
T_EVAL=T[ev]; SC_EVAL=clf.predict_proba(X[ev])[:,1]
EVAL_EVENTS=catalog[(catalog.breach_utc >= START_EVAL+pd.Timedelta(minutes=LEAD_MAX))&
                    (catalog.breach_utc<END_EVAL)].copy().reset_index(drop=True)
assert len(EVAL_EVENTS)>0
# The 24-hour refractory in the catalog should make alert windows for distinct events disjoint.
assert np.all(np.diff(EVAL_EVENTS.breach_utc.astype('int64'))>=24*60*NS)
print('TRAIN bars:',int(train.sum()),'positive:',int(label[train].sum()))
print('CAL bars:',len(T_CAL),'EVAL bars:',len(T_EVAL),'EVAL events:',len(EVAL_EVENTS))
print('Important: zero-volume regularized bars are excluded from scoring and execution.')

TRAIN bars: 524087 positive: 18779
CAL bars: 525600 EVAL bars: 1750857 EVAL events: 83
Important: zero-volume regularized bars are excluded from scoring and execution.


## 2. Four schedulers: same score, causal decisions, annual hard cap

In [4]:
BUDGETS=[60,120,240,480]; MODES=['calendar','rolling30','paced','token_bucket']
PRIMARY_B=240

# Silence timezone warnings from to_period in the helper by using UTC-naive dates internally.
def time_features(ts_ns):
    idx=pd.to_datetime(np.asarray(ts_ns,dtype='int64'),utc=True)
    months=(idx.year.to_numpy()*12+idx.month.to_numpy()).astype('int32')
    years=idx.year.to_numpy(dtype='int32')
    naive=idx.tz_localize(None)
    first=naive.to_period('M').to_timestamp()
    after=first+pd.offsets.MonthBegin(1)
    frac=np.asarray((naive-first)/pd.Timedelta(minutes=1),dtype=float) / np.asarray((after-first)/pd.Timedelta(minutes=1),dtype=float)
    return months,years,frac

def token_capacity(B): return max(2,int(math.ceil(B/48)))

def emit_policy(t, scores, thresh, B, mode, burst=None, detailed=False):
    """At each contemporaneous score crossing, decide to emit or discard; no event label used."""
    t=np.asarray(t,dtype='int64');scores=np.asarray(scores,dtype='float64')
    assert len(t)==len(scores) and np.all(np.diff(t)>0)
    assert mode in MODES and B>0 and B%12==0
    mon,yr,frac=time_features(t)
    cap=B//12
    if burst is None:burst=token_capacity(B)
    if burst<1:raise ValueError('bucket capacity must be >=1')
    # Continuous refill: nominal B tokens per 365 days, initial full bucket.
    refill_per_min=B/(365.0*24*60)
    candidate=np.flatnonzero(np.isfinite(scores)&(scores>=float(thresh)))
    out=[];prev_alert=-10**21;prev_year=-1;prev_month=-1
    count_year=0;count_month=0
    rolling=deque();tokens=float(burst);last_token_time=int(t[0]) if len(t) else 0
    detail=defaultdict(lambda:{'score_crossings':0,'alerts':0,'annual_blocks':0,'monthly_blocks':0,
          'pacing_blocks':0,'rolling_blocks':0,'token_blocks':0,'cooldown_blocks':0,
          'exhausted_utc':None,'first7_alerts':0})
    for i in candidate:
        ti=int(t[i]);mi=int(mon[i]);yi=int(yr[i]);dt=detail[mi]
        dt['score_crossings']+=1
        if yi!=prev_year:
            prev_year=yi;count_year=0
        if mi!=prev_month:
            prev_month=mi;count_month=0
        if mode=='token_bucket':
            elapsed=max(0,(ti-last_token_time)/NS)
            tokens=min(float(burst),tokens+elapsed*refill_per_min)
            last_token_time=ti
        if count_year>=B:
            dt['annual_blocks']+=1;continue
        if mode=='calendar' or mode=='paced':
            if count_month>=cap:
                dt['monthly_blocks']+=1;continue
            if mode=='paced':
                allowance=min(cap, 2+int(np.floor(cap*float(frac[i]))))
                if count_month>=allowance:
                    dt['pacing_blocks']+=1;continue
        elif mode=='rolling30':
            while rolling and ti-rolling[0]>=30*24*60*NS:rolling.popleft()
            if len(rolling)>=cap:
                dt['rolling_blocks']+=1;continue
        elif mode=='token_bucket':
            if tokens<1.0-1e-10:
                dt['token_blocks']+=1;continue
        if ti-prev_alert<COOLDOWN_MIN*NS:
            dt['cooldown_blocks']+=1;continue
        out.append(ti);prev_alert=ti;count_year+=1;count_month+=1;dt['alerts']+=1
        if frac[i] < 7/31: dt['first7_alerts']+=1  # approximate; exact first7 available below
        if mode=='rolling30':rolling.append(ti)
        if mode=='token_bucket':tokens=max(0.,tokens-1.)
        if mode in ('calendar','paced') and count_month==cap and dt['exhausted_utc'] is None:
            dt['exhausted_utc']=ti
    out=np.asarray(out,dtype='int64')
    if detailed:return out,dict(detail)
    return out

# Unit tests. A non-monotone count-vs-threshold is possible under cooldown: scan, don't binary search.
toy=pd.date_range('2022-01-01',periods=12,freq='h',tz='UTC').asi8
for mode in MODES:
    al=emit_policy(toy,np.ones(len(toy)),.5,60,mode)
    assert np.all(np.diff(al)>=COOLDOWN_MIN*NS),mode
assert len(emit_policy(toy,np.zeros(len(toy)),.5,60,'calendar'))==0
assert len(emit_policy(toy,np.ones(len(toy)),.5,60,'calendar'))==3
assert len(emit_policy(toy,np.ones(len(toy)),.5,60,'rolling30'))==3
assert len(emit_policy(toy,np.ones(len(toy)),.5,60,'paced'))<=3
print('Schedulers: preliminary unit tests passed')

Schedulers: preliminary unit tests passed


## 3. Event matching, monthly saturation and the exact offline oracle

In [5]:
def match_events(events,alerts):
    a=np.sort(np.asarray(alerts,dtype='int64'))
    claimed=set();rows=[]
    for e in events.itertuples(index=False):
        b=int(e.breach_utc.value);on=int(e.onset_1pct_utc.value)
        l=int(np.searchsorted(a,b-LEAD_MAX*NS,'left'))
        r=int(np.searchsorted(a,b-LEAD_MIN*NS,'right'))
        choices=[j for j in range(l,r) if j not in claimed]
        if not choices:continue
        j=choices[0];claimed.add(j)
        rows.append(dict(event_id=int(e.event_id), breach_utc=e.breach_utc,
             alarm_utc=pd.Timestamp(int(a[j]),tz='UTC'),lead_min=(b-int(a[j]))/NS,
             before_1pct=bool(a[j]<on)))
    return pd.DataFrame(rows,columns=['event_id','breach_utc','alarm_utc','lead_min','before_1pct'])

def score_stats(events,alerts):
    m=match_events(events,alerts)
    n=len(alerts);n_event=len(events)
    return dict(alarms=n,events=n_event,caught=len(m),pre_onset_caught=int(m.before_1pct.sum()),
        false_alarms=n-len(m),event_recall=len(m)/n_event,
        pre_onset_recall=int(m.before_1pct.sum())/n_event,
        event_precision=len(m)/n if n else 0,
        avg_lead_min=float(m.lead_min.mean()) if len(m) else np.nan)

# A hindsight event-placement oracle. For each price event, list only candidate score-crossing
# timestamps that exist inside its [30,240] min window, grouped by alert month.
# This is a true exact optimum for its stated binary integer model, but is a clairvoyant ceiling,
# not an implementable policy. All events must be separated enough to avoid cooldown conflicts.
def make_oracle_options(events,t,score,threshold):
    t=np.asarray(t,dtype='int64');score=np.asarray(score,dtype=float)
    assert np.all(np.diff(t)>0)
    hit=t[np.isfinite(score)&(score>=threshold)]
    opts=[];eligible_ids=[];early_ids=[]
    for e in events.itertuples(index=False):
        b=e.breach_utc.value;on=e.onset_1pct_utc.value
        L=np.searchsorted(hit,b-LEAD_MAX*NS,'left')
        R=np.searchsorted(hit,b-LEAD_MIN*NS,'right')
        candidates=hit[L:R]
        if len(candidates)==0:continue
        eligible_ids.append(int(e.event_id))
        if (candidates<on).any():early_ids.append(int(e.event_id))
        months,years,_=time_features(candidates)
        for m in np.unique(months):
            ix=(months==m)
            c=candidates[ix];y=int(years[np.flatnonzero(ix)[0]])
            opts.append(dict(event=int(e.event_id),month=int(m),year=y,early=bool((c<on).any()),
                        candidate_ns=int(c[0]),count_candidates=int(len(c))))
    return pd.DataFrame(opts,columns=['event','month','year','early','candidate_ns','count_candidates']),eligible_ids,early_ids

def oracle_optimal(options,B,monthly=False,strict_preonset=False):
    """Max unique events with event- and annual-cap constraints; optional month cap.
    Exact integer program over event×calendar-month options, because 24h event refractory
    ensures cooldown conflicts among different event windows cannot occur.
    """
    x=options[options.early].copy() if strict_preonset else options.copy()
    if len(x)==0:return 0
    ev_ids=sorted(x.event.unique());mons=sorted(x.month.unique());years=sorted(x.year.unique())
    rows=[];upper=[]
    for e in ev_ids:
        rows.append(np.asarray((x.event==e).astype(float)));upper.append(1)
    for y in years:
        rows.append(np.asarray((x.year==y).astype(float)));upper.append(B)
    if monthly:
        for m in mons:
            rows.append(np.asarray((x.month==m).astype(float)));upper.append(B//12)
    A=sparse.csr_matrix(np.vstack(rows))
    con=LinearConstraint(A,np.full(len(upper),-np.inf),np.asarray(upper,float))
    sol=milp(c=-np.ones(len(x)),integrality=np.ones(len(x)),
         bounds=Bounds(np.zeros(len(x)),np.ones(len(x))),constraints=con,
         options={'mip_rel_gap':0.0,'time_limit':120})
    if sol.status!=0 or sol.x is None:
        raise RuntimeError(f'Oracle MILP did not certify optimality; status={sol.status}: {sol.message}')
    selected=int(np.rint(sol.x).sum())
    return selected

# Exact Oracle sanity: 3 events, one month with two events, calendar cap=1 at B=12.
_o=pd.DataFrame([{'event':1,'month':2023*12+1,'year':2023,'early':True,'candidate_ns':1,'count_candidates':1},
    {'event':2,'month':2023*12+1,'year':2023,'early':True,'candidate_ns':2,'count_candidates':1},
    {'event':3,'month':2023*12+2,'year':2023,'early':False,'candidate_ns':3,'count_candidates':1}])
assert oracle_optimal(_o,B=12,monthly=True)==2
assert oracle_optimal(_o,B=2,monthly=False)==2
assert oracle_optimal(_o,B=12,monthly=True,strict_preonset=True)==1
print('Event matching and exact MILP oracle functions ready')

Event matching and exact MILP oracle functions ready


## 4. Calibration on 2022 only (no outcome labels)

In [6]:
# Quantile grid predeclared. Calendar baseline threshold is shared across all schedulers in
# the fixed-score arm. Recalibration arm scans thresholds separately for each scheduler.
QUANTILES=np.unique(np.r_[0.,np.linspace(.10,.995,33),.997,.999,.9999,1.])
CAL_THRESHOLDS=np.unique(np.r_[np.nextafter(SC_CAL.min(),-np.inf),
                     np.quantile(SC_CAL,QUANTILES),np.nextafter(SC_CAL.max(),np.inf)])

def find_threshold_for_B(mode,B):
    res=[(float(th),len(emit_policy(T_CAL,SC_CAL,float(th),B,mode))) for th in CAL_THRESHOLDS]
    # Avoid over-target alert counts where equally close.
    chosen=min(res,key=lambda p:(abs(p[1]-B),p[1]>B,-p[0]))
    return chosen[0],chosen[1],max(k for _,k in res)

cal_rows=[]
for B in BUDGETS:
    for mode in MODES:
        th,n,maximum=find_threshold_for_B(mode,B)
        cal_rows.append(dict(B=B,mode=mode,threshold_2022=th,alarms_2022=n,
             calibration_error=n-B,max_reachable_2022=maximum,
             burst_size=token_capacity(B) if mode=='token_bucket' else np.nan))
cal_tbl=pd.DataFrame(cal_rows)
cal_tbl.to_csv(OUT/'threshold_calibration_2022.csv',index=False)
display(cal_tbl.round(6))
assert cal_tbl.groupby('B').size().eq(len(MODES)).all()
print('Calibration reads no 2023+ scores or outcomes; the calibration error is in the table below.')

,B,mode,threshold_2022,alarms_2022,calibration_error,max_reachable_2022,burst_size
0,60,calendar,0.716607,60,0,60,NaN
1,60,rolling30,0.669899,60,0,60,NaN
2,60,paced,0.716607,60,0,60,NaN
3,60,token_bucket,0.669899,60,0,60,2.0
4,120,calendar,0.716607,120,0,120,NaN
5,120,rolling30,0.639706,120,0,120,NaN
6,120,paced,0.669899,120,0,120,NaN
7,120,token_bucket,0.669899,120,0,120,3.0
8,240,calendar,0.669899,240,0,240,NaN
9,240,rolling30,0.616149,240,0,240,NaN


Calibration reads no 2023+ scores or outcomes; the calibration error is in the table below.


## 5. Policy evaluation, diagnostic decomposition and oracle ceilings

In [7]:
# Token-bucket burst-size sensitivity uses 2022 only. The primary burst size is
# engineering-predeclared (ceil(B/48)) rather than selected from repeatedly inspected evaluation outcomes.
burst_rows=[]
for B in [120,PRIMARY_B,480]:
    base_thr=float(cal_tbl[(cal_tbl.B==B)&(cal_tbl['mode']=='token_bucket')].threshold_2022.iloc[0])
    events_2022=catalog[(catalog.breach_utc>=START_CAL+pd.Timedelta(minutes=LEAD_MAX)) &
                        (catalog.breach_utc<START_EVAL)]
    for b in sorted(set([2,5,10,20,token_capacity(B)])):
        trial=emit_policy(T_CAL,SC_CAL,base_thr,B,'token_bucket',burst=b)
        stats=score_stats(events_2022,trial)
        burst_rows.append(dict(B=B,burst_capacity=b,threshold_2022=base_thr,**stats))
burst_sensitivity=pd.DataFrame(burst_rows)
burst_sensitivity.to_csv(OUT/'DEV_ONLY_bucket_burst_sensitivity_2022.csv',index=False)
print('Token-bucket burst sensitivity, development period only:')
display(burst_sensitivity[burst_sensitivity.B==PRIMARY_B][['B','burst_capacity','alarms','caught','pre_onset_caught']])

# Include late Dec 2022 when warming up rolling/token schedulers; only score 2023+ alerts.
warm_mask=T_CAL>=pd.Timestamp('2022-12-01',tz='UTC').value
T_WARM=np.r_[T_CAL[warm_mask],T_EVAL]
SC_WARM=np.r_[SC_CAL[warm_mask],SC_EVAL]
assert np.all(np.diff(T_WARM)>0)

policies=[];monthly=[];matches=[];oracles=[];raw_alerts={};annual=[]
for B in BUDGETS:
    base_th=float(cal_tbl[(cal_tbl.B==B)&(cal_tbl['mode']=='calendar')].threshold_2022.iloc[0])
    for mode in MODES:
        chosen=float(cal_tbl[(cal_tbl.B==B)&(cal_tbl['mode']==mode)].threshold_2022.iloc[0])
        for arm,th in [('fixed_threshold',base_th),('recalibrated',chosen)]:
            alerts,details=emit_policy(T_WARM,SC_WARM,th,B,mode,detailed=True)
            alerts=alerts[alerts>=START_EVAL.value]
            raw_alerts[(B,mode,arm)]=alerts
            metrics=score_stats(EVAL_EVENTS,alerts)
            policies.append(dict(B=B,mode=mode,arm=arm,threshold_2022=th,**metrics))
            mat=match_events(EVAL_EVENTS,alerts)
            if len(mat):matches.append(mat.assign(B=B,mode=mode,arm=arm))
            # Monthly consumption, including zero-alert months.
            for month in pd.period_range('2023-01','2026-04',freq='M'):
                mid=month.year*12+month.month
                z=details.get(mid,{})
                ms=pd.Timestamp(str(month),tz='UTC');me=ms+pd.offsets.MonthBegin(1)
                month_alerts=alerts[(alerts>=ms.value)&(alerts<me.value)]
                exhausted=z.get('exhausted_utc',None)
                monthly.append(dict(B=B,mode=mode,arm=arm,month=str(month),
                   alarms=len(month_alerts),monthly_target=B//12,
                   first_7d_alerts=int((month_alerts<ms.value+7*24*60*NS).sum()),
                   score_crossings=z.get('score_crossings',0),
                   blocked_annual=z.get('annual_blocks',0),
                   blocked_monthly=z.get('monthly_blocks',0),
                   blocked_pacing=z.get('pacing_blocks',0),
                   blocked_rolling=z.get('rolling_blocks',0),
                   blocked_token=z.get('token_blocks',0),
                   blocked_cooldown=z.get('cooldown_blocks',0),
                   quota_exhausted_utc=pd.Timestamp(exhausted,tz='UTC') if exhausted else pd.NaT,
                   days_to_exhaustion=(exhausted-ms.value)/NS/1440 if exhausted else np.nan,
                   month_days_left=(me.value-exhausted)/NS/1440 if exhausted else np.nan))
            for year in range(2023,2027):
                start=pd.Timestamp(f'{year}-01-01',tz='UTC').value
                end=pd.Timestamp(f'{year+1}-01-01',tz='UTC').value
                annual.append(dict(B=B,mode=mode,arm=arm,year=year,
                        alarms=int(((alerts>=start)&(alerts<end)).sum()),hard_annual_cap=B))
            # Signal-conditional, exact clairvoyant ceilings for each threshold.
            # These offline optimizations see future outcomes; never train or deploy them.
            options,reachable,early=make_oracle_options(EVAL_EVENTS,T_EVAL,SC_EVAL,th)
            if B==PRIMARY_B and mode=='calendar' and arm=='recalibrated':
                options.to_csv(OUT/'primary_oracle_event_month_options.csv',index=False)
            oracle_row=dict(B=B,mode=mode,arm=arm,threshold_2022=th,
                signal_reachable=len(reachable),signal_preonset_reachable=len(early),
                annual_clairvoyant_hits=oracle_optimal(options,B,monthly=False),
                annual_clairvoyant_preonset=oracle_optimal(options,B,monthly=False,strict_preonset=True),
                calendar_clairvoyant_hits=oracle_optimal(options,B,monthly=True),
                calendar_clairvoyant_preonset=oracle_optimal(options,B,monthly=True,strict_preonset=True))
            oracles.append(oracle_row)
res=pd.DataFrame(policies);sat=pd.DataFrame(monthly);matched=pd.concat(matches,ignore_index=True) if matches else pd.DataFrame()
oracle=pd.DataFrame(oracles);yearly=pd.DataFrame(annual)
res.to_csv(OUT/'scheduler_policy_comparison.csv',index=False)
sat.to_csv(OUT/'alert_saturation_by_month.csv',index=False)
matched.to_csv(OUT/'matched_events_by_scheduler.csv',index=False)
oracle.to_csv(OUT/'hindsight_oracle_ceilings.csv',index=False)
yearly.to_csv(OUT/'alerts_by_calendar_year.csv',index=False)
assert (yearly.alarms<=yearly.hard_annual_cap).all(), 'Annual budget violation'
no_alert=pd.DataFrame([dict(B=0,mode='no_alert',arm='no_alert',alarms=0,events=len(EVAL_EVENTS),caught=0,pre_onset_caught=0,false_alarms=0,event_recall=0,pre_onset_recall=0,event_precision=0)])
no_alert.to_csv(OUT/'no_alert_baseline.csv',index=False)
print('Retrospective policy results / 2022-recalibrated arms:')
display(res[res.arm=='recalibrated'].round(4).sort_values(['B','mode']))
print('Hindsight oracle (future-aware upper bound, not implementable; the gap is not only scheduler inefficiency):')
display(oracle[(oracle.B==PRIMARY_B)&(oracle.arm=='recalibrated')])

Token-bucket burst sensitivity, development period only:


,B,burst_capacity,alarms,caught,pre_onset_caught
5,240,2,225,15,7
6,240,5,240,13,6
7,240,10,240,14,6
8,240,20,240,15,7


Retrospective policy results / 2022-recalibrated arms:


,B,mode,arm,threshold_2022,alarms,events,caught,pre_onset_caught,false_alarms,event_recall,pre_onset_recall,event_precision,avg_lead_min
1,60,calendar,recalibrated,0.7166,200,83,11,6,189,0.1325,0.0723,0.0550,130.2727
5,60,paced,recalibrated,0.7166,199,83,10,4,189,0.1205,0.0482,0.0503,95.8000
3,60,rolling30,recalibrated,0.6699,196,83,6,1,190,0.0723,0.0120,0.0306,63.5000
7,60,token_bucket,recalibrated,0.6699,199,83,7,1,192,0.0843,0.0120,0.0352,52.1429
9,120,calendar,recalibrated,0.7166,400,83,26,11,374,0.3133,0.1325,0.0650,122.2308
13,120,paced,recalibrated,0.6699,399,83,13,5,386,0.1566,0.0602,0.0326,106.3077
11,120,rolling30,recalibrated,0.6397,393,83,17,5,376,0.2048,0.0602,0.0433,85.1176
15,120,token_bucket,recalibrated,0.6699,395,83,13,4,382,0.1566,0.0482,0.0329,89.7692
17,240,calendar,recalibrated,0.6699,800,83,39,22,761,0.4699,0.2651,0.0488,134.5128
21,240,paced,recalibrated,0.6397,789,83,28,14,761,0.3373,0.1687,0.0355,115.4286


Hindsight oracle (future-aware upper bound, not implementable; the gap is not only scheduler inefficiency):


,B,mode,arm,threshold_2022,signal_reachable,signal_preonset_reachable,annual_clairvoyant_hits,annual_clairvoyant_preonset,calendar_clairvoyant_hits,calendar_clairvoyant_preonset
17,240,calendar,recalibrated,0.669899,68,39,68,39,68,39
19,240,rolling30,recalibrated,0.616149,77,56,77,56,77,56
21,240,paced,recalibrated,0.639706,75,48,75,48,75,48
23,240,token_bucket,recalibrated,0.639706,75,48,75,48,75,48


## 6. Paired comparisons, monthly cluster diagnostics and statistical uncertainty

In [ ]:
paired=[];EIDS=set(EVAL_EVENTS.event_id.astype(int))
for B in BUDGETS:
    for arm in ['fixed_threshold','recalibrated']:
        a0=raw_alerts[(B,'calendar',arm)]
        m0=match_events(EVAL_EVENTS,a0)
        old_ids=set(m0.event_id);old_pre=set(m0.loc[m0.before_1pct.astype(bool),'event_id'])
        for mode in ['paced','rolling30','token_bucket']:
            a1=raw_alerts[(B,mode,arm)];m1=match_events(EVAL_EVENTS,a1)
            new_ids=set(m1.event_id);new_pre=set(m1.loc[m1.before_1pct.astype(bool),'event_id'])
            only_old=len(old_ids-new_ids);only_new=len(new_ids-old_ids)
            old_p=len(old_pre-new_pre);new_p=len(new_pre-old_pre)
            paired.append(dict(B=B,arm=arm,comparison='calendar_vs_'+mode,
              calendar_alerts=len(a0),alternative_alerts=len(a1),delta_alerts=len(a1)-len(a0),
              calendar_caught=len(old_ids),alternative_caught=len(new_ids),
              delta_caught=len(new_ids)-len(old_ids),
              calendar_preonset=len(old_pre),alternative_preonset=len(new_pre),
              delta_preonset=len(new_pre)-len(old_pre),
              only_calendar=only_old,only_alt=only_new,
              paired_exact_p=binomtest(only_old,only_old+only_new).pvalue if only_old+only_new else 1.0,
              only_calendar_pre=old_p,only_alt_pre=new_p,
              paired_preonset_p=binomtest(old_p,old_p+new_p).pvalue if old_p+new_p else 1.0))
paired=pd.DataFrame(paired)
paired.to_csv(OUT/'paired_scheduler_comparison.csv',index=False)
display(paired[(paired.B==PRIMARY_B)].round(4))

# Primary hypothesis: monthly crash clustering and monthly score-trigger load clustering.
# In-sample diagnostics on development (2021-2022), separate retrospective 2023+ descriptions.
cat_month=catalog[(catalog.breach_utc>=START_TRAIN)&(catalog.breach_utc<END_EVAL)].copy()
for name,start,end in [('development','2021-01','2022-12'),('retrospective','2023-01','2026-04')]:
    months=pd.period_range(start,end,freq='M')
    breaks=cat_month[cat_month.breach_utc.dt.to_period('M').isin(months)]
    counts=breaks.breach_utc.dt.tz_localize(None).dt.to_period('M').value_counts().reindex(months,fill_value=0).sort_index()
    disp=counts.var(ddof=1)/counts.mean() if counts.mean()>0 else np.nan
    print(f'{name}: {len(months)} months, {int(counts.sum())} events, monthly mean {counts.mean():.3f}, variance/mean {disp:.3f} (not automatically Hawkes evidence)')

# Plot only policy results pre-selected by model and predefined budgets; no ex-post winner selection.
z=res[res.arm=='recalibrated'].sort_values('alarms')
fig,ax=plt.subplots(figsize=(9,5))
for mode,g in z.groupby('mode'):
    ax.plot(g.alarms,g.pre_onset_caught,marker='o',label=mode)
    for row in g.itertuples():ax.annotate(str(row.B),(row.alarms,row.pre_onset_caught),xytext=(3,4),textcoords='offset points',fontsize=8)
ax.set(xlabel='Realized alerts (2023–Apr 2026)',ylabel='Events alerted before first 1% decline',
       title='Scheduling trade-off: single frozen V+F score')
ax.grid(alpha=.2);ax.legend();fig.tight_layout();fig.savefig(OUT/'scheduler_preonset_frontier.png',dpi=170);plt.show()

ss=sat[(sat.B==PRIMARY_B)&(sat['mode']=='calendar')&(sat.arm=='recalibrated')]
fig,ax=plt.subplots(figsize=(10,3.5))
ax.bar(np.arange(len(ss)),ss.month_days_left.fillna(0))
ax.set(xlabel='Months from Jan 2023 to Apr 2026',ylabel='Days left after quota exhausted',
       title='Calendar alert budget exhaustion; zeros include nonexhausted months')
ax.set_xticks(range(0,len(ss),4));ax.set_xticklabels(ss.month.iloc[::4],rotation=40,ha='right')
fig.tight_layout();fig.savefig(OUT/'calendar_month_saturation.png',dpi=170);plt.show()

# Bootstrap event-level monthly differences in the primary arm by contiguous 3-month blocks.
# Moving-block bootstrap is descriptive, especially with repeated exploration and nonstationarity.
def preonset_by_month(B,mode,arm):
    m=match_events(EVAL_EVENTS,raw_alerts[(B,mode,arm)])
    months=pd.period_range('2023-01','2026-04',freq='M').astype(str)
    if len(m)==0 or not bool(m.before_1pct.astype(bool).any()):return np.zeros(len(months),dtype=float)
    c=pd.to_datetime(m.loc[m['before_1pct'].astype(bool),'breach_utc'],utc=True).dt.tz_localize(None).dt.to_period('M').astype(str).value_counts()
    return np.array([c.get(x,0) for x in months],dtype=float)

rng=np.random.default_rng(1009);bootstrap=[]
for mode in ['paced','rolling30','token_bucket']:
    change=preonset_by_month(PRIMARY_B,mode,'recalibrated')-preonset_by_month(PRIMARY_B,'calendar','recalibrated')
    n=len(change);repl=2000;block=3
    sample=np.empty(repl)
    for j in range(repl):
        chosen=[]
        while len(chosen)<n:
            p=int(rng.integers(0,n-block+1));chosen.extend(range(p,p+block))
        sample[j]=change[np.array(chosen[:n])].sum()
    lo,hi=np.quantile(sample,[.025,.975])
    bootstrap.append(dict(B=PRIMARY_B,mode=mode,primary_preonset_delta=int(change.sum()),
                          bootstrap_low=float(lo),bootstrap_high=float(hi),months=n,block_months=block))
boot=pd.DataFrame(bootstrap);boot.to_csv(OUT/'block_bootstrap_primary_delta.csv',index=False)
display(boot)

## 6b. Crash clustering versus false-alarm demand

In [9]:
# Calendar and raw-score metrics are deliberately separated. Raw threshold crossings at every
# minute are *not independent alarms*; look at emitted false alerts for the operational test.
# Crash months have different counts and may share exogenous volatility; this is an association
# audit, not a causal proof that alert demand is highest exactly when crashes cluster.
anchor_th=float(cal_tbl[(cal_tbl.B==PRIMARY_B)&(cal_tbl['mode']=='calendar')].threshold_2022.iloc[0])

def forward_event_label_for_ns(tt):
    z=np.searchsorted(BREACH,tt+LEAD_MIN*NS,'left')
    ok=z<len(BREACH)
    q=np.zeros(len(tt),dtype=bool)
    q[ok]=(BREACH[z[ok]]<=tt[ok]+LEAD_MAX*NS)
    return q

candidates=T_EVAL[np.isfinite(SC_EVAL)&(SC_EVAL>=anchor_th)]
candidates_true=forward_event_label_for_ns(candidates)
all_months=pd.period_range('2023-01','2026-04',freq='M')
months_candidate=pd.to_datetime(candidates,utc=True).tz_localize(None).to_period('M')
raw_counts=pd.Series(months_candidate).value_counts().reindex(all_months,fill_value=0)
raw_false=pd.Series(months_candidate[~candidates_true]).value_counts().reindex(all_months,fill_value=0)
crashes=(EVAL_EVENTS.breach_utc.dt.tz_localize(None).dt.to_period('M')
         .value_counts().reindex(all_months,fill_value=0))
# Calendar baseline actual alerts and their matched, one-to-one event hits.
cal_alarms=raw_alerts[(PRIMARY_B,'calendar','recalibrated')]
cal_matches=match_events(EVAL_EVENTS,cal_alarms)
true_alarms=set(cal_matches.alarm_utc.astype('int64').to_numpy().tolist())
false_alarms_ns=np.array([x for x in cal_alarms if int(x) not in true_alarms],dtype='int64')
false_months=(pd.Series(pd.to_datetime(false_alarms_ns,utc=True).tz_localize(None).to_period('M'))
               .value_counts().reindex(all_months,fill_value=0))

def spearman_safe(x,y):
    return spearmanr(x,y) if len(set(x))>1 and len(set(y))>1 else (np.nan,np.nan)

corr=[]
for metric_label,vals in [('raw_crossings',raw_counts),('raw_false_crossings',raw_false),('emitted_false_alerts',false_months)]:
    rho,p=spearman_safe(crashes.to_numpy(),vals.to_numpy())
    corr.append(dict(metric=metric_label,months=len(all_months),spearman_rho=rho,
                     exploratory_p=p,model='V+F ungated, calendar, primary budget',
                     important='same-month association is not causal; repeated exploratory evaluation'))
corr=pd.DataFrame(corr)
monthly_co=pd.DataFrame(dict(month=all_months.astype(str),crashes=crashes.values,
                  raw_crossings=raw_counts.values,raw_false_crossings=raw_false.values,
                  emitted_false_alerts=false_months.values))
monthly_co.to_csv(OUT/'co_clustering_monthly_audit.csv',index=False)
corr.to_csv(OUT/'co_clustering_descriptive_correlations.csv',index=False)
print('False-alert versus crash burden association (descriptive only):')
display(corr.round(4))

False-alert versus crash burden association (descriptive only):


,metric,months,spearman_rho,exploratory_p,model,important
0,raw_crossings,40,0.4686,0.0023,"V+F ungated, calendar, primary budget",same-month association is not causal; repeated...
1,raw_false_crossings,40,0.3922,0.0123,"V+F ungated, calendar, primary budget",same-month association is not causal; repeated...
2,emitted_false_alerts,40,-0.7361,0.0000,"V+F ungated, calendar, primary budget",same-month association is not causal; repeated...


## 6c. Expanding-window walk-forward validation (exploratory; annual refit and previous-year calibration)

In [ ]:
# Fixed V+F feature specification throughout; 2021 train / 2022 calibration / 2023 test,
# 2021-22 train / 2023 calibration / 2024 test, and so on.
# The terminal 245 minutes of each training split are embargoed to stop event lookahead.
# Policy thresholds calibrated on prior-year scores only, never prior-year crash labels.
# This is not a fresh confirmation: 2023+ has already been investigated multiple times.
wf_rows=[];wf_match=[]
for year in [2023,2024,2025,2026]:
    begin_cal=pd.Timestamp(f'{year-1}-01-01',tz='UTC')
    begin_test=pd.Timestamp(f'{year}-01-01',tz='UTC')
    end_test=min(pd.Timestamp(f'{year+1}-01-01',tz='UTC'),END_EVAL)
    tr=valid&(T>=START_TRAIN.value)&(T<(begin_cal-pd.Timedelta(minutes=LEAD_MAX+5)).value)
    ca=valid&(T>=begin_cal.value)&(T<begin_test.value)
    te=valid&(T>=begin_test.value)&(T<end_test.value)
    if tr.sum()==0 or ca.sum()==0 or te.sum()==0:raise ValueError(f'Incomplete WF split {year}')
    mod=make_pipeline(StandardScaler(),LogisticRegression(C=.2,class_weight='balanced',
                          max_iter=400,solver='lbfgs',random_state=17))
    mod.fit(X[tr],label[tr].astype('int8'))
    ca_t=T[ca];ca_score=mod.predict_proba(X[ca])[:,1]
    te_t=T[te];te_score=mod.predict_proba(X[te])[:,1]
    cal_grid=np.unique(np.r_[np.nextafter(ca_score.min(),-np.inf),
                np.quantile(ca_score,QUANTILES),np.nextafter(ca_score.max(),np.inf)])
    y_events=catalog[(catalog.breach_utc>=begin_test+pd.Timedelta(minutes=LEAD_MAX)) &
                     (catalog.breach_utc<end_test)].copy()
    for mode in MODES:
        grid=[(float(th),len(emit_policy(ca_t,ca_score,th,PRIMARY_B,mode))) for th in cal_grid]
        th,cal_n=min(grid,key=lambda v:(abs(v[1]-PRIMARY_B),v[1]>PRIMARY_B,-v[0]))
        # Prime 30d rolling history and token bucket from previous-year *causal scores*.
        t_all=np.r_[ca_t,te_t];s_all=np.r_[ca_score,te_score]
        generated=emit_policy(t_all,s_all,th,PRIMARY_B,mode)
        result=generated[generated>=begin_test.value]
        pair=match_events(y_events,result)
        if len(pair):wf_match.append(pair.assign(test_year=year,mode=mode))
        metr=score_stats(y_events,result)
        wf_rows.append(dict(test_year=year,train_start='2021-01-01',train_end_exclusive=str(begin_cal),
                 calibrate_year=year-1,mode=mode,annual_target=PRIMARY_B,
                 calibrated_threshold=th,calibration_alerts=int(cal_n),**metr))
wf=pd.DataFrame(wf_rows)
wf.to_csv(OUT/'EXPLORATORY_expanding_walkforward_annual.csv',index=False)
if wf_match:
    pd.concat(wf_match,ignore_index=True).to_csv(OUT/'EXPLORATORY_walkforward_event_matches.csv',index=False)
print('Expanding walk-forward; 2023+ retrospective, not independent confirmation:')
display(wf.sort_values(['test_year','mode'])[['test_year','mode','calibration_alerts','alarms','caught','pre_onset_caught']])

Expanding walk-forward; 2023+ retrospective, not independent confirmation:


,test_year,mode,calibration_alerts,alarms,caught,pre_onset_caught
0,2023,calendar,240,240,7,3
2,2023,paced,240,240,4,2
1,2023,rolling30,240,240,4,2
3,2023,token_bucket,240,240,4,2
4,2024,calendar,240,240,13,6
6,2024,paced,240,237,11,4
5,2024,rolling30,240,232,12,2
7,2024,token_bucket,240,240,12,2
8,2025,calendar,240,240,11,8
10,2025,paced,240,240,5,3


## 7. Decision layer: what is an alert worth?

After each emitted alert I reduce a 100%-long BTC exposure by Δw = 0.5 for 240 minutes, trading at the next valid one-minute close, and charge a one-way fee `c` when I trim the exposure and again when I restore it; overlapping 4-hour windows merge into one reduced-exposure spell. I compare compounded wealth with buy-and-hold and a constant exposure-matched benchmark. Financing costs, funding, market impact and short borrowing lie outside the model, so this is an illustrative decision analysis, not a deployable strategy.

Break-even precision: if a matched alert avoids an average loss `L>0` over the 4-hour hedge window, an unmatched alert gives up an expected 4-hour gain `g`, and each round trip costs `2c`, then

$$
E[\Delta W]/\Delta w = pL-(1-p)g-2c,\qquad
p^*=\frac{g+2c}{L+g}.
$$

Neither the 4% drawdown label nor the event-based precision equals the actual 4-hour avoidable loss or the conditional probability, so I evaluate realized hedged returns directly.

In [ ]:
# Prepare *contiguous* 1m price path, not just scoring-eligible bars. Ignore synthetic zero-volume
# for execution; price series can be used as a reference index but zero-volume is not executable.
CLOSE_TIMES=T;CLOSE=close;VOL=volume
# Spot archives can contain exchange-halt gaps longer than 5 minutes, which the cache keeps as NaN.
# Returns use the last observed close, so a gap's price move lands on the first bar after reopening
# (buy-and-hold total is preserved). Execution still requires a real positive-volume bar (CLOSE/VOL unchanged).
# Without gaps these are plain close-to-close returns.
_CLOSE_PATH=pd.Series(CLOSE).ffill().bfill().to_numpy(dtype=float)
print('Minutes with missing close (long gaps):',int((~np.isfinite(CLOSE)).sum()))
RETS=np.zeros(len(CLOSE),dtype=float)
RETS[1:]=_CLOSE_PATH[1:]/_CLOSE_PATH[:-1]-1
if not np.isfinite(RETS).all():raise ValueError('Nonfinite BTC close-to-close returns')

HOLD_MIN=240;DW=0.5;PRIMARY_FEE_BPS=10.;FEE_GRID=(0.,5.,10.,25.)
EVAL_INDEX=np.flatnonzero((CLOSE_TIMES>=START_EVAL.value)&(CLOSE_TIMES<END_EVAL.value))
assert len(EVAL_INDEX)>0

# Signal at bar close t; execution at first strictly following *positive-volume* minute close,
# with max 5 minutes' execution delay. Position affects only subsequent bar close returns.
def hedge_spells(alarm_times,horizon=HOLD_MIN,max_execution_wait=5):
    a=np.asarray(alarm_times,dtype='int64')
    spells=[];dropped=0
    for t in a:
        i=int(np.searchsorted(CLOSE_TIMES,t,'right'))
        if i>=len(CLOSE_TIMES): dropped+=1;continue
        end_search=min(i+max_execution_wait,len(CLOSE_TIMES))
        good=np.flatnonzero((VOL[i:end_search]>0)&np.isfinite(CLOSE[i:end_search])&(CLOSE[i:end_search]>0))
        if len(good)==0:dropped+=1;continue
        entry=i+int(good[0]);exit_target=entry+horizon
        if exit_target>=len(CLOSE_TIMES) or CLOSE_TIMES[exit_target]>=END_EVAL.value:
            dropped+=1;continue
        # Close the hedge as close as possible to target, within 5 min on a positive-volume bar.
        j=min(exit_target+max_execution_wait,len(CLOSE_TIMES))
        exits=np.flatnonzero((VOL[exit_target:j]>0)&np.isfinite(CLOSE[exit_target:j])&(CLOSE[exit_target:j]>0))
        if not len(exits):dropped+=1;continue
        exit_i=exit_target+int(exits[0])
        spells.append((entry,exit_i))
    # Union overlapping hedges; fees charged for real exposure transitions only.
    spells.sort()
    merged=[]
    for st,en in spells:
        if merged and st<=merged[-1][1]: merged[-1]=(merged[-1][0],max(merged[-1][1],en))
        else:merged.append((st,en))
    return merged,dropped

def simulate_action(alarms,fee_bps=PRIMARY_FEE_BPS):
    spells,dropped=hedge_spells(alarms)
    lo=int(EVAL_INDEX[0]);hi=int(EVAL_INDEX[-1])+1
    exposure=np.ones(hi-lo,dtype=float)
    charges=np.zeros(hi-lo,dtype=float)
    one_way=float(fee_bps)*1e-4
    for st,en in spells:
        # At entry close, first reduced-exposure return is the following minute.
        first=st+1;last=en
        if first<hi and last>=lo:
            exposure[max(lo,first)-lo:min(hi,last+1)-lo]=1-DW
        if lo<=st<hi: charges[st-lo]+=DW*one_way
        if lo<=en<hi: charges[en-lo]+=DW*one_way
    r=RETS[lo:hi]
    policy_step=exposure*r-charges
    exposure_mean=float(exposure.mean())
    hold_step=r
    matched_step=exposure_mean*r
    for s in [policy_step,hold_step,matched_step]:
        if np.any(s<=-1):raise ValueError('Gross bar return <=-100%')
    log_policy=np.log1p(policy_step);log_hold=np.log1p(hold_step);log_matched=np.log1p(matched_step)
    cumulative=np.exp(np.cumsum(log_policy))
    mdd=float(np.min(cumulative/np.maximum.accumulate(np.r_[1.,cumulative])[1:]-1))
    idx=btc.index[lo:hi]
    monthly_adv=(pd.Series(log_policy-log_matched,index=idx)
           .groupby(idx.to_period('M')).sum())
    return dict(alarms=len(alarms),executed_spells=len(spells),dropped_unfillable=dropped,
       one_way_fee_bps=fee_bps,mean_exposure=exposure_mean,turnover_roundtrips=len(spells),
       net_terminal_return=float(np.expm1(log_policy.sum())),
       buyhold_terminal_return=float(np.expm1(log_hold.sum())),
       exposure_matched_terminal_return=float(np.expm1(log_matched.sum())),
       incremental_logwealth_vs_exposure_matched=float((log_policy-log_matched).sum()),
       incremental_logwealth_vs_buyhold=float((log_policy-log_hold).sum()),
       max_drawdown=mdd),monthly_adv

# Use 2022 thresholds selected before testing later outcomes; primary fee 10bp is fixed by spec.
rows=[];adv_months=[]
for B in BUDGETS:
    for mode in MODES:
        for fee in FEE_GRID:
            alarms=raw_alerts[(B,mode,'recalibrated')]
            x,month_delta=simulate_action(alarms,fee)
            rows.append(dict(B=B,mode=mode,arm='recalibrated',**x))
            if B==PRIMARY_B and fee==PRIMARY_FEE_BPS:
                adv_months.append(pd.DataFrame({'month':month_delta.index.astype(str),
                   'incremental_logwealth':month_delta.values,'mode':mode}))
econ=pd.DataFrame(rows);monthly_econ=pd.concat(adv_months,ignore_index=True)
econ.to_csv(OUT/'economic_decision_sensitivity.csv',index=False)
monthly_econ.to_csv(OUT/'monthly_exposure_matched_value.csv',index=False)
print('Illustrative 10bp economic decision analysis (not live trading):')
display(econ[(econ.B==PRIMARY_B)&(econ.one_way_fee_bps==PRIMARY_FEE_BPS)].round(5))

# Moving 3-calendar-month block bootstrap of log-wealth advantage vs exposure-matched comparator.
rng=np.random.default_rng(2026);ci=[]
for mode in MODES:
    vals=(monthly_econ[monthly_econ['mode']==mode].sort_values('month')
        .incremental_logwealth.to_numpy(dtype=float))
    n=len(vals);draws=[];block=3
    for k in range(2000):
        sel=[]
        while len(sel)<n:
            p=int(rng.integers(0,n-block+1));sel.extend(range(p,p+block))
        draws.append(vals[np.array(sel[:n])].sum())
    q=np.quantile(draws,[.025,.975])
    ci.append(dict(mode=mode,B=PRIMARY_B,fee_bps=PRIMARY_FEE_BPS,
        incremental_logwealth=float(vals.sum()),lower_95=float(q[0]),upper_95=float(q[1]),
        block_months=block,draws=2000))
econ_ci=pd.DataFrame(ci);econ_ci.to_csv(OUT/'economic_month_block_bootstrap.csv',index=False)
display(econ_ci.round(6))

# Unit checks on timing and zero-return economics (temporarily replace return series; restore).
zero_alarm_stats,_=simulate_action(np.array([],dtype='int64'),PRIMARY_FEE_BPS)
assert abs(zero_alarm_stats['incremental_logwealth_vs_buyhold'])<1e-10
candidate_time=int(T_EVAL[len(T_EVAL)//2])
sp,skipped=hedge_spells(np.array([candidate_time],dtype='int64'))
if sp:
    assert CLOSE_TIMES[sp[0][0]]>candidate_time, 'Same-minute fill (lookahead) detected'
    assert sp[0][1]-sp[0][0]>=HOLD_MIN
    _old_rets=RETS.copy()
    try:
        RETS[:]=0.0
        flat,_=simulate_action(np.array([candidate_time],dtype='int64'),PRIMARY_FEE_BPS)
        assert flat['net_terminal_return']<0, 'Roundtrip fees should lose money when returns are zero'
    finally:
        RETS[:]=_old_rets
print('Execution alignment, no-alert benchmark and fee-only economic tests passed.')

Minutes with missing close (long gaps): 2325


/tmp/ipykernel_297/4222970487.py:71: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .groupby(idx.to_period('M')).sum())
/tmp/ipykernel_297/4222970487.py:71: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .groupby(idx.to_period('M')).sum())
/tmp/ipykernel_297/4222970487.py:71: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .groupby(idx.to_period('M')).sum())
/tmp/ipykernel_297/4222970487.py:71: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .groupby(idx.to_period('M')).sum())
/tmp/ipykernel_297/4222970487.py:71: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .groupby(idx.to_period('M')).sum())


Illustrative 10bp economic decision analysis (not live trading):


,B,mode,arm,alarms,executed_spells,dropped_unfillable,one_way_fee_bps,mean_exposure,turnover_roundtrips,net_terminal_return,buyhold_terminal_return,exposure_matched_terminal_return,incremental_logwealth_vs_exposure_matched,incremental_logwealth_vs_buyhold,max_drawdown
34,240,calendar,recalibrated,800,717,0,10.0,0.94518,717,1.89528,3.61521,3.32991,-0.40247,-0.46628,-0.51615
38,240,rolling30,recalibrated,791,748,0,10.0,0.94579,748,0.98496,3.61521,3.33307,-0.78068,-0.84376,-0.53813
42,240,paced,recalibrated,789,780,0,10.0,0.94593,780,0.77843,3.61521,3.33378,-0.89071,-0.95363,-0.57114
46,240,token_bucket,recalibrated,793,784,0,10.0,0.94566,784,0.58379,3.61521,3.33237,-1.00630,-1.06954,-0.61024


,mode,B,fee_bps,incremental_logwealth,lower_95,upper_95,block_months,draws
0,calendar,240,10.0,-0.402466,-0.594073,-0.080582,3,2000
1,rolling30,240,10.0,-0.780679,-1.039592,-0.449747,3,2000
2,paced,240,10.0,-0.890709,-1.128139,-0.591414,3,2000
3,token_bucket,240,10.0,-1.006296,-1.325113,-0.735335,3,2000


/tmp/ipykernel_297/4222970487.py:71: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .groupby(idx.to_period('M')).sum())


Execution alignment, no-alert benchmark and fee-only economic tests passed.


## 8. Break-even precision from calibration-period 4-hour returns (descriptive, noisy)

In [ ]:
# Breakeven inputs must refer to the *same* stylized 4h hedge action, not the 4% crash label.
# A warning may be a hit by the 4% breach criterion while 4h post-alert return is positive.
cal_events=catalog[(catalog.breach_utc>=START_CAL+pd.Timedelta(minutes=LEAD_MAX)) &
                   (catalog.breach_utc<START_EVAL)].copy()
be=[]
for B in BUDGETS:
    for mode in MODES:
        threshold=float(cal_tbl[(cal_tbl.B==B)&(cal_tbl['mode']==mode)].threshold_2022.iloc[0])
        a=emit_policy(T_CAL,SC_CAL,threshold,B,mode)
        matched_ids=match_events(cal_events,a)
        truth_alert_times=set(matched_ids.alarm_utc.astype('int64').to_list())
        costs=[];hits=[]
        for t in a:
            i=int(np.searchsorted(CLOSE_TIMES,t,'right'))
            if i>=len(CLOSE_TIMES):continue
            good=np.flatnonzero(VOL[i:min(i+5,len(VOL))]>0)
            if not len(good):continue
            st=i+int(good[0]);en=st+HOLD_MIN
            if en>=len(CLOSE_TIMES) or CLOSE_TIMES[en]>=START_EVAL.value:continue
            ret=float(CLOSE[en]/CLOSE[st]-1.0)
            costs.append(ret);hits.append(t in truth_alert_times)
        costs=np.asarray(costs,float);hits=np.asarray(hits,bool)
        p=float(hits.mean()) if len(hits) else np.nan
        L=float(-np.mean(costs[hits])) if hits.any() else np.nan
        g=float(np.mean(costs[~hits])) if (~hits).any() else np.nan
        for fee_bps in FEE_GRID:
            c=fee_bps*1e-4
            denominator=L+g
            pstar=(g+2*c)/denominator if np.isfinite(denominator) and denominator>0 else np.nan
            be.append(dict(B=B,mode=mode,fee_oneway_bps=fee_bps,
                cal_alerts_with_valid_4h=len(costs),cal_event_prec=p,
                avoidable_4h_loss_if_matched=L,nonmatched_4h_mean_return=g,
                breakeven_event_precision=pstar,
                admissible_probability=bool(np.isfinite(pstar) and 0<=pstar<=1),
                warning='Small matched samples; scenario/identity not a reliable forward forecast'))
be=pd.DataFrame(be);be.to_csv(OUT/'calibration_break_even_precision.csv',index=False)
display(be[(be.B==PRIMARY_B)&(be.fee_oneway_bps==PRIMARY_FEE_BPS)].round(5))
print('Interpret event precision as a count-based proxy, not a calibrated conditional probability.')

,B,mode,fee_oneway_bps,cal_alerts_with_valid_4h,cal_event_prec,avoidable_4h_loss_if_matched,nonmatched_4h_mean_return,breakeven_event_precision,admissible_probability,warning
34,240,calendar,10.0,240,0.07083,0.02595,-0.00150,0.02062,True,Small matched samples; scenario/identity not a...
38,240,rolling30,10.0,240,0.05417,0.03772,0.00025,0.05923,True,Small matched samples; scenario/identity not a...
42,240,paced,10.0,240,0.06250,0.02699,-0.00046,0.05790,True,Small matched samples; scenario/identity not a...
46,240,token_bucket,10.0,240,0.05417,0.02650,0.00069,0.09902,True,Small matched samples; scenario/identity not a...


Interpret event precision as a count-based proxy, not a calibrated conditional probability.


## 9. Clustering and ODE input to scheduler design (optional; not a crash-prediction signal)

In [ ]:
# Calendar-month count statistics and homogeneous-Poisson null on the 2021-2022 development period.
# Variance/mean >1 is not evidence of Hawkes self-excitation by itself; structural regime
# changes, seasonality, and nonstationary intensity can generate the same overdispersion.
months=pd.period_range('2021-01','2022-12',freq='M')
evt=catalog[(catalog.breach_utc>=START_TRAIN)&(catalog.breach_utc<START_EVAL)]
cnt=evt.breach_utc.dt.tz_localize(None).dt.to_period('M').value_counts().reindex(months,fill_value=0).sort_index()
mean=cnt.mean();vmr=cnt.var(ddof=1)/mean
rng=np.random.default_rng(44)
sim=rng.poisson(mean,size=(20000,len(cnt)))
sim_vmr=sim.var(axis=1,ddof=1)/np.maximum(sim.mean(axis=1),1e-12)
pois_p=float((sim_vmr>=vmr).mean())
disp=pd.DataFrame([dict(period='2021-2022 only',n_months=len(cnt),events=int(cnt.sum()),
            mean_monthly=float(mean),variance_monthly=float(cnt.var(ddof=1)),
            index_dispersion=float(vmr),homogeneous_poisson_MC_p=pois_p)])
disp.to_csv(OUT/'development_crash_clustering.csv',index=False)
display(disp)

# Optional exponential Hawkes point-process MLE for event *occurrence times* (days).
# lambda(t)=mu+alpha * sum_{ti<t} exp(-beta(t-ti)); branching ratio=alpha/beta in (0,1).
# An ODE obeyed by the excitation state: dZ/dt=-beta Z + sum delta(t-ti).
# This is a descriptive development-period model, not a verified value-of-information result.
times=(evt.breach_utc.astype('int64').to_numpy()-START_TRAIN.value)/(24*60*NS)
times=np.sort(times);horizon=(START_EVAL-START_TRAIN)/pd.Timedelta(days=1)

def hawkes_nll(par):
    logmu,logbeta,logitn=par
    mu=np.exp(logmu);beta=np.exp(logbeta);n=1/(1+np.exp(-logitn))
    alpha=beta*n
    q=0.;previous=0.;ll=0.
    for ti in times:
        q*=np.exp(-beta*(ti-previous))
        intensity=mu+alpha*q
        ll+=np.log(intensity)
        q+=1.;previous=ti
    integrated=mu*horizon+n*np.sum(1-np.exp(-beta*(horizon-times)))
    return float(integrated-ll)

if len(times)>=10:
    guesses=[np.array([np.log(max(.005,len(times)/horizon/2)),np.log(1/3),-1.]),
             np.array([np.log(max(.005,len(times)/horizon)),np.log(1/10),-2.])]
    fits=[minimize(hawkes_nll,g,method='L-BFGS-B',bounds=[(-9,5),(-9,7),(-9,3)]) for g in guesses]
    fit=min(fits,key=lambda z:z.fun)
    mu=np.exp(fit.x[0]);beta=np.exp(fit.x[1]);branch=1/(1+np.exp(-fit.x[2]))
    poi_mu=len(times)/horizon
    poisson_nll=poi_mu*horizon-len(times)*np.log(poi_mu)
    clustering=pd.DataFrame([dict(n_events=len(times),hawkes_success=bool(fit.success),
          hawkes_mu_per_day=mu,decay_beta_per_day=beta,branching_ratio=branch,
          mean_excitation_life_days=1/beta,hawkes_AIC=2*3+2*fit.fun,
          homogeneous_poisson_AIC=2+2*poisson_nll,
          limitation='preliminary nonstationary data; not proof of self-excitation')])
    clustering.to_csv(OUT/'EXPLORATORY_hawkes_development_only.csv',index=False)
    display(clustering.round(4))
else:
    print('Not enough crash events for descriptive Hawkes estimate')

,period,n_months,events,mean_monthly,variance_monthly,index_dispersion,homogeneous_poisson_MC_p
0,2021-2022 only,24,144,6.0,21.043478,3.507246,0.0


,n_events,hawkes_success,hawkes_mu_per_day,decay_beta_per_day,branching_ratio,mean_excitation_life_days,hawkes_AIC,homogeneous_poisson_AIC,limitation
0,144,True,0.0837,0.1129,0.5756,8.8537,734.9154,757.4906,preliminary nonstationary data; not proof of s...


## 10. Interpretation and outputs

In [ ]:
print(f'Spot evaluation events: {len(EVAL_EVENTS)} (futures data: 84).')
assert res.groupby(['B','arm']).size().eq(len(MODES)).all()
assert (res.false_alarms>=0).all()
assert (res.caught<=res.alarms).all()
assert (res.pre_onset_caught<=res.caught).all()
assert (oracle.annual_clairvoyant_hits>=oracle.calendar_clairvoyant_hits).all()
assert (oracle.annual_clairvoyant_preonset>=oracle.calendar_clairvoyant_preonset).all()

report=[]
report.append('# BTC Systems Engineering Research — Retrospective Findings (Auto-generated)')
report.append('')
report.append('All 2023–Apr 2026 findings are EXPLORATORY; their periods were previously inspected.')
report.append('Budget levels are experimental controls, not economically optimal choices. Oracle is clairvoyant, not implementable.')
report.append('')
report.append('## Primary budget: B=240 annual alerts (experimental)')
report.append('')
report.append(res[(res.B==PRIMARY_B)&(res.arm=='recalibrated')][
        ['mode','alarms','caught','pre_onset_caught','false_alarms','event_recall','pre_onset_recall']]
        .to_markdown(index=False))
report.append('')
report.append('## Paired primary outcome delta against calendar cap')
report.append('')
report.append(paired[(paired.B==PRIMARY_B)&(paired.arm=='recalibrated')][
        ['comparison','delta_alerts','delta_caught','delta_preonset','paired_exact_p','paired_preonset_p']]
        .to_markdown(index=False))
report.append('')
report.append('## Interpretation reminders')
report.append('')
report.extend(['- Alarm counts are compared as realized burdens; equality in 2022 calibration does not force equality in 2023+.',
               '- No-alert remains an operational reference; 0 alerts and 0 events caught.',
               '- Oracle can see future events and discard false alerts, so oracle gap is NOT attributable only to scheduling.',
               '- Event-matching precision is not the probability of avoided 4-hour loss.',
               '- All economic outcomes reflect simplified proxy execution, not actual trading returns.',
               '- A development-period SHA-256 written today does NOT retroactively preregister earlier data exploration.',
               '- Keep the 4% independent event catalog fixed; 3%/5% are separately labeled sensitivity analyses.'])
(OUT/'RETROSPECTIVE_TECHNICAL_SUMMARY.md').write_text('\n'.join(report),encoding='utf-8')
print('Output directory:',OUT)
for path in sorted(OUT.iterdir()):
    if path.is_file():print(' -',path.name)
print('Decision rule: prioritize pre-onset event detection at similar REALIZED alert count; report uncertainty and nulls.')